# HyperDrone

`hyperdrone.env.MultiEnvironment` is the C++ `rl_tools` environment used by the training
targets — vision in the loop, batch-first, JIT-compiled on first construction. The
environment is one object with four levels of ownership — **use it, configure it, drive
it, or extend it** — and this notebook walks one drone through all four:

1. **Fly the preset** — a platform is one config field.
2. **A task is one knob** — shipped tasks cost one config field, not new bindings.
3. **Your own task is a Python loop** — the verbs are caller-driven; reward shaping,
   curricula, and termination logic need zero C++.
4. **Your own world is a small C++ header** — derive-and-shadow the same specification
   the presets use; the JIT compiles it like any other config.

In [ ]:
import importlib.util
if importlib.util.find_spec("hyperdrone") is None:  # fresh runtime (e.g. Colab); skipped in a dev checkout
    !sudo apt-get update -qq && sudo apt-get install -y -qq libassimp-dev
    !pip install -q "hyperdrone[examples]"

## Setup

The environment schedules over a directory of `.glb` scenes: every scene in it is
loaded up front and kept resident with its own renderer and free-space table — that is
what makes per-episode scene rotation instant and deterministic — so the directory is
the working set, and staging it (symlinks are fine) is how you select the scenes to
train over. The default below stages a single downloaded ProcTHOR scene; set
`HYPERDRONE_SCENES` to a directory of scenes (e.g. a ProcTHOR corpus) to stage the
first `NUM_SCENES` of them.

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from hyperdrone.env import EnvConfig, MultiEnvironment
from hyperdrone.examples.data import procthor_scene_path, x500_model_path

NUM_SCENES = 8
scenes = Path("scenes")
if not scenes.exists():
    source = os.environ.get("HYPERDRONE_SCENES")
    staged = sorted(Path(source).glob("*.glb"))[:NUM_SCENES] if source else [Path(procthor_scene_path())]
    scenes.mkdir()
    for path in staged:
        (scenes / path.name).symlink_to(path)
drone_asset = x500_model_path()

def show(images, titles, size=3):
    figure, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for axis, image, title in zip(np.atleast_1d(axes), images, titles):
        axis.imshow(np.clip(image, 0, 1), interpolation="nearest")
        axis.set_title(title, fontsize=9)
        axis.axis("off")
    plt.tight_layout()
    plt.show()

## 1. Fly the preset

A preset names the platform (`presets::*` on the C++ side): `x500_fpv` is an FPV drone
whose own frame and props are visible to its camera, with the prop rotation driven by
the rotor dynamics (`StateRenderRotorPhase` → the rigged `prop_*` nodes of the GLB
assembly) — holding the motors at mid command below keeps the body still for one step,
so between the two consecutive frames only the prop pixels change.

This is the exact environment behind the C++ training targets: the same batch verbs
(`reset` / `render` / `observe` / `step` / `rewards` / `terminated`), the same
implementation — a seeded rollout here is bit-exact against the C++ verbs (pinned by a
cross-language golden test). The first construction JIT-compiles the configuration and
caches it.

In [ ]:
env = MultiEnvironment(
    scenes,
    config=EnvConfig(instances=4, cam_width=64, cam_height=64, preset="x500_fpv"),
    seed=0,
    drone_asset=drone_asset,
)
print(env.config_string)

reset_all = np.ones(env.total_instances, dtype=np.uint8)
reset_none = np.zeros(env.total_instances, dtype=np.uint8)
env.reset(reset_all)
env.render(reset_all)
hold = np.zeros((env.total_instances, env.action_dim), dtype=np.float32)  # motors at mid command
history = [env.frames()]
for _ in range(3):
    env.step(hold)
    env.render(reset_none)
    history.append(env.frames())

show(list(history[-1]), [f"instance {i} (FPV)" for i in range(env.total_instances)])
show([history[-2][0], history[-1][0]], ["step n", "step n+1 — the props have advanced"])

## 2. A task is one knob

Tasks are C++ wrappers around the world (`tasks::*`). Flipping `task="target_frame"`
extends the observation with the target frame to reach — the named `observation_layout`
shows exactly what changed, no magic indices.

In [ ]:
task_env = MultiEnvironment(
    scenes,
    config=EnvConfig(instances=4, cam_width=64, cam_height=64, preset="x500_fpv", task="target_frame"),
    seed=0,
    drone_asset=drone_asset,
)
print("preset only: ", env.observation_layout)
print("target_frame:", task_env.observation_layout)

task_env.reset(reset_all)
task_env.render(reset_all)
frame = task_env.frames()[0]
show(
    [frame[..., offset:offset + size] for offset, size in task_env.observation_layout.blocks.values()],
    list(task_env.observation_layout.blocks),
)
task_env.close()

## 3. Your own task is a Python loop

The verbs are caller-driven, and the presets keep the base MDP neutral — the reward is
zero until a task (like `moving_gate`) or you define one, so `rewards()` /
`terminated()` are defaults you may use or ignore. Read the state by named block from
the privileged observation, compute your own reward and termination, and feed your
reset mask back into `reset()` / `render()` — reward shaping, curricula, and
termination logic need zero C++.

In [ ]:
layout = env.observation_layout_privileged
print(layout.blocks)

def block(state, name):
    offset, size = layout.blocks[name]
    return state[:, offset:offset + size]

target = np.array([0.0, 0.0, 1.0], dtype=np.float32)
rng = np.random.default_rng(0)
env.reset(reset_all)
env.render(reset_all)
for _ in range(20):
    env.step(rng.uniform(-1, 1, size=(env.total_instances, env.action_dim)).astype(np.float32))
    state = env.observe_privileged()
    distance = np.linalg.norm(block(state, "position") - target, axis=1)
    reward = -distance - 0.1 * np.linalg.norm(block(state, "linear_velocity"), axis=1)  # yours
    terminated = distance > 2.0                                                        # yours
    env.reset(terminated.astype(np.uint8))
    env.render(terminated.astype(np.uint8))
print("custom rewards:     ", np.round(reward, 3))
print("custom terminations:", terminated)
env.close()

## 4. Your own world is a small C++ header

When a custom task needs what Python can't express — camera optics, motion blur, new
observation channels, scene entities, a compiled reward — you write the same
derive-and-shadow specification the presets use and hand it to the JIT via
`spec_header=`. The header pins the World, so `preset` / `task` / `n_agents` stay at
their defaults; its content hash is part of the JIT key, so editing it recompiles. The
shipped presets (`presets.h`) and tasks (`tasks/`) are your fork templates. One
difference: a user World's observation layout is exported as a single flat block (the
shim cannot name your blocks), so reshape the observation yourself.

In [ ]:
%%writefile my_world.h
#pragma once
#include <rl_tools/rl/environments/hyperdrone/presets.h>
#include <cstddef>

namespace hyperdrone_env_user {
    using T = float;
    using TI = std::size_t;
    struct SPEC: rl_tools::rl::environments::hyperdrone::presets::X500FPV<T, TI> {
        static constexpr TI INSTANCES_PER_ENVIRONMENT = 4;
        static constexpr TI CAM_WIDTH = 64;
        static constexpr TI CAM_HEIGHT = 64;
        using SHADING = rl_tools::rendering::raytracing::Medium;
        static constexpr bool ENABLE_MOTION_BLUR = true;
        static constexpr TI MOTION_BLUR_SAMPLES = 8;
        static constexpr T CAMERA_FOV = 1.6;
    };
    using WORLD = rl_tools::rl::environments::hyperdrone::World<SPEC>;
}

In [ ]:
custom = MultiEnvironment(scenes, config=EnvConfig(spec_header="my_world.h"), seed=0, drone_asset=drone_asset)
print(custom.config_string)

reset_all = np.ones(custom.total_instances, dtype=np.uint8)
custom.reset(reset_all)
custom.render(reset_all)
for _ in range(3):
    custom.step(rng.uniform(-1, 1, size=(custom.total_instances, custom.action_dim)).astype(np.float32))
    custom.render(np.zeros(custom.total_instances, dtype=np.uint8))
views = custom.observe().reshape(custom.total_instances, custom.cam_height, custom.cam_width, custom.image_channels)
show(list(views), [f"instance {i}" for i in range(custom.total_instances)])
custom.close()

For rung 4 done seriously — a user-authored task wrapper with a moving object, compiled
reward, and collision termination — see `Orbiter.ipynb`. For manual composition of the
renderer and dynamics (external cameras, foundation-policy flight), see `Custom.ipynb`;
for throughput measurement, `Benchmark.ipynb`.